# División temporal de los datos

Los datos se dividen respetando el orden cronológico para evitar que el modelo aprenda información del futuro

- Entrenamiento: hasta diciembre de 2022
- Validación: 2023 y 2024
- Prueba final: desde enero de 2025
- Predicción futura: última jornada sin objetivo conocido

In [1]:
# aqui solo cargo el dataset

from pathlib import Path 
import pandas as pd

# Localizo la carpeta principal
ruta_actual = Path.cwd().resolve()

if ruta_actual.name == "notebooks":
    ruta_proyecto = ruta_actual.parent
else:
    ruta_proyecto = ruta_actual

ruta_processed = ruta_proyecto / "data" / "processed"

archivo_variables = (
    ruta_processed / "eurusd_variables_modelo.csv"
)

datos = pd.read_csv(
    archivo_variables,
    parse_dates=["Date"],
    index_col="Date"
)

datos["Objetivo"] = datos["Objetivo"].astype("Int64") # uso Int64 (es un tipo especial de pandas) para guardar enteros
                                                      # y no tener ningun problemas con valores nulos (N/A)

print("Dimensiones:", datos.shape)
print("Primera fecha:", datos.index.min().date())
print("Última fecha:", datos.index.max().date())

Dimensiones: (5825, 20)
Primera fecha: 2004-01-29
Última fecha: 2026-07-14


In [2]:
# La última fila no tiene objetivo porque aún no existe la jornada siguiente
datos_futuro = datos[
    datos["Objetivo"].isna()
].copy()

# utilizo únicamente las filas que ya conozco el resutlad ya que el modelo necesita esa respuesta para aprender y para comprobar 
# posteriormente si sus predicciones fueron correctas
datos_historicos = datos[
    datos["Objetivo"].notna()
].copy()

datos_historicos["Objetivo"] = (
    datos_historicos["Objetivo"].astype(int) #los vuelvo a cambiar a enteros normales para no tener problemas despues
)

print("Filas históricas:", len(datos_historicos))
print("Filas para predicción futura:", len(datos_futuro))

Filas históricas: 5824
Filas para predicción futura: 1


In [3]:
FECHA_FIN_ENTRENAMIENTO = "2022-12-31"
FECHA_FIN_VALIDACION = "2024-12-31" 

# Estas fechas dejan un periodo amplio para entrenar, dos años completos para tomar decisiones y los datos más recientes para comprobar 
# el resultado final

In [4]:
entrenamiento = datos_historicos.loc[
    datos_historicos.index <= FECHA_FIN_ENTRENAMIENTO
].copy()

validacion = datos_historicos.loc[
    (datos_historicos.index > FECHA_FIN_ENTRENAMIENTO)
    & (datos_historicos.index <= FECHA_FIN_VALIDACION)
].copy()

prueba = datos_historicos.loc[
    datos_historicos.index > FECHA_FIN_VALIDACION
].copy()

#Como los datos siguen un orden temporal, no utilizo train_test_split ya que podría mezclar fechas antiguas con fechas futuras
# Por eso separo directamente los periodos por fecha para que el modelo aprenda con el pasado y se evalúe con información posterior

In [5]:
resumen_particiones = pd.DataFrame({
    "filas": [
        len(entrenamiento),
        len(validacion),
        len(prueba),
        len(datos_futuro)
    ],
    "fecha_inicio": [
        entrenamiento.index.min().date(),
        validacion.index.min().date(),
        prueba.index.min().date(),
        datos_futuro.index.min().date()
    ],
    "fecha_fin": [
        entrenamiento.index.max().date(),
        validacion.index.max().date(),
        prueba.index.max().date(),
        datos_futuro.index.max().date()
    ]
}, index=[
    "Entrenamiento",
    "Validación",
    "Prueba",
    "Predicción futura"
])

display(resumen_particiones)

#Esta tabla me permite ver de forma rápida cómo quedaron repartidos los datos entre entrenamiento, validación y prueba final, 
# indicando las fechas y la cantidad de filas de cada periodo (mas adelante me srive para explicar y justificar la división 
# que usé en el proyecto)

,filas,fecha_inicio,fecha_fin
Entrenamiento,4908,2004-01-29,2022-12-30
Validación,522,2023-01-02,2024-12-31
Prueba,394,2025-01-02,2026-07-13
Predicción futura,1,2026-07-14,2026-07-14


In [6]:
if (
    len(entrenamiento)
    + len(validacion)
    + len(prueba)
    != len(datos_historicos)
):
    raise ValueError(
        "Algunas filas históricas no fueron asignadas correctamente."
    )

if not (
    entrenamiento.index.max()
    < validacion.index.min()
    < prueba.index.min()
):
    raise ValueError(
        "Las particiones no respetan el orden temporal."
    )

print("División temporal realizada correctamente")

# aqui confirmo dos cosas
 # Que todas las filas históricas estén dentro de entrenamiento, validación o prueba
 # Que las fechas mantengan el orden correcto: primero entrenamiento, luego validación y después prueba

División temporal realizada correctamente


In [7]:
entrenamiento["Particion"] = "entrenamiento"
validacion["Particion"] = "validacion"
prueba["Particion"] = "prueba"
datos_futuro["Particion"] = "futuro"

datos_particionados = pd.concat([
    entrenamiento,
    validacion,
    prueba,
    datos_futuro
]).sort_index()

#Antes separé las filas históricas que sí tienen un objetivo conocido, de la última fila destinada a la predicción futura. Después dividí 
# las filas históricas en entrenamiento, validación y prueba según la fecha. En esta parte añado una etiqueta a cada grupo y los vuelvo a unir 
# en un solo dataset ordenado, para conservar claramente a qué partición pertenece cada fila

In [8]:
archivo_particiones = (
    ruta_processed / "eurusd_particiones.csv"
)

datos_particionados.to_csv(
    archivo_particiones,
    index=True,
    encoding="utf-8"
)

print("Archivo guardado en:")
print(archivo_particiones)

display(
    datos_particionados[
        ["Close", "Objetivo", "Particion"]
    ].tail()
)

Archivo guardado en:
C:\TFM_EURUSD\data\processed\eurusd_particiones.csv


,Close,Objetivo,Particion
Date,,,
2026-07-08,1.140381,1,prueba
2026-07-09,1.142204,1,prueba
2026-07-10,1.143341,0,prueba
2026-07-13,1.140446,0,prueba
2026-07-14,1.138433,<NA>,futuro
